In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. INITIALISATION ET GÉNÉRATION DES DONNÉES (Votre code)
# ---------------------------------------------------------
produits = pd.DataFrame({
    "Code_produit": ["P001","P002","P003","P004","P005"],
    "Designation": ["Paracetamol","Amoxicilline","Quinine","SRO","Ceftriaxone"],
    "Forme": ["Comprime","Capsule","Injectable","Sachet","Injectable"],
    "DCI": ["Paracetamol","Amoxicilline","Quinine","Sels de rehydratation","Ceftriaxone"],
    "Dosage": ["500mg","500mg","300mg","20.5g","1g"],
    "Unite": ["Boite","Boite","Ampoule","Sachet","Flacon"],
    "Prix_unitaire_FCFA": [500,1200,800,300,2500],
    "Fournisseur_principal": ["Lab A","Lab B","Lab C","Lab D","Lab B"],
    "Delai_reapprovisionnement_jours": [30,45,40,20,50]
})






# 2. ANALYSE ET CALCULS (Votre code optimisé)
# ---------------------------------------------------------








# ---------------------------------------------------------
# 3. CRÉATION DU DASHBOARD (Subplots)
# ---------------------------------------------------------







In [ ]:
dates = pd.date_range(end=datetime.today(), periods=180)
mouvements = []
depots = ["Central","Ouaga","Bobo","Ouahigouya"]
motifs = ["Achat","Vente","Peremption","Casse"]
vendeur = ["Francis","alizata", "Aminata","Delfine", "Diane", "Siaka", "Latif"]



In [ ]:
# Calcul du stock actuel
mouvements_stock["Stock_actuel"] = mouvements_stock["Quantite"] * mouvements_stock["Type"].map({"Entree": 1, "Sortie": -1})
stock_actuel = mouvements_stock.groupby(["Code_produit", "Depot"])["Stock_actuel"].sum().reset_index()# Calcul du stock actuel
mouvements_stock["Stock_actuel"] = mouvements_stock["Quantite"] * mouvements_stock["Type"].map({"Entree": 1, "Sortie": -1})
stock_actuel = mouvements_stock.groupby(["Code_produit", "Depot"])["Stock_actuel"].sum().reset_index()

In [ ]:
# CMM et Stock de sécurité
sorties = mouvements_stock[mouvements_stock["Type"]=="Sortie"].copy()
cmm = sorties.groupby("Code_produit")["Quantite"].sum() / 6
cmm = cmm.reset_index().rename(columns={"Quantite":"CMM"})
cmm["Stock_securite"] = 2 * cmm["CMM"]

In [ ]:
# Niveaux de commande
produits_cmm = produits.merge(cmm, on="Code_produit")
produits_cmm["consommation_moyenne_journaliere"] = produits_cmm["CMM"] / 30
produits_cmm["Niveau_commande"] = produits_cmm["Stock_securite"] + (produits_cmm["Delai_reapprovisionnement_jours"] * produits_cmm["consommation_moyenne_journaliere"])

In [ ]:
# Statut du stock
stock = stock_actuel.merge(produits_cmm, on="Code_produit")
def statut_stock(row):
    if row["Stock_actuel"] <= 0: return "Rupture"
    elif row["Stock_actuel"] < row["Stock_securite"]: return "Alerte critique"
    elif row["Stock_actuel"] < row["Niveau_commande"]: return "Alerte"
    elif row["Stock_actuel"] > 12 * row["CMM"]: return "Surstock"
    else: return "Stock correct"

stock["Statut"] = stock.apply(statut_stock, axis=1)


In [ ]:
# Valorisation et Pertes
stock["Valeur_stock"] = stock["Stock_actuel"] * stock["Prix_unitaire_FCFA"]
valeur_depot = stock.groupby("Depot")["Valeur_stock"].sum()

sorties["Mois"] = sorties["Date"].dt.to_period("M")
pertes = mouvements_stock[mouvements_stock["Motif"].isin(["Peremption","Casse"])]
# Ajout de la désignation pour un affichage plus lisible sur le graphique
pertes = pertes.merge(produits[["Code_produit", "Designation", "Prix_unitaire_FCFA"]], on="Code_produit")
pertes["Cout"] = pertes["Quantite"] * pertes["Prix_unitaire_FCFA"]


In [ ]:
# Configuration globale de la figure
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
fig.suptitle('Tableau de Bord - Gestion des Stocks Pharmaceutiques', fontsize=18, fontweight='bold', y=0.98)


In [ ]:
# --- Subplot 1 : Valeur des stocks par dépôt (Bar chart) ---
# Filtrer les valeurs négatives éventuelles causées par la génération aléatoire
valeur_depot = valeur_depot[valeur_depot > 0] 
valeur_depot.plot(kind='bar', ax=axes[0, 0], color='#4C72B0', edgecolor='black')
axes[0, 0].set_title('Valeur Financière des Stocks par Dépôt (FCFA)', fontsize=14)
axes[0, 0].set_ylabel('Valeur Totale (FCFA)')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=0)
axes[0, 0].grid(axis='y', linestyle='--', alpha=0.7)


In [ ]:
# --- Subplot 2 : Répartition du statut des stocks (Pie chart) ---
statut_counts = stock['Statut'].value_counts()
couleurs_statut = {'Stock correct': '#55A868', 'Surstock': '#8172B3', 'Alerte': '#F1CE63', 'Alerte critique': '#E8A75D', 'Rupture': '#C44E52'}
couleurs_pie = [couleurs_statut.get(x, '#999999') for x in statut_counts.index]

axes[0, 1].pie(statut_counts, labels=statut_counts.index, autopct='%1.1f%%', startangle=140, colors=couleurs_pie, wedgeprops={'edgecolor': 'black'})
axes[0, 1].set_title('État de Santé du Stock', fontsize=14)


In [ ]:
# --- Subplot 3 : Tendance des sorties mensuelles (Line chart) ---
sorties_mensuelles = sorties.groupby('Mois')['Quantite'].sum()
sorties_mensuelles.index = sorties_mensuelles.index.astype(str) # Convertir la période en texte pour l'affichage
axes[1, 0].plot(sorties_mensuelles.index, sorties_mensuelles.values, marker='o', color='#C44E52', linewidth=2, markersize=8)
axes[1, 0].set_title('Tendance des Sorties Mensuelles (Toutes causes)', fontsize=14)
axes[1, 0].set_ylabel('Quantité Totale Sortie')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, linestyle='--', alpha=0.7)


In [ ]:
# --- Subplot 4 : Coût des pertes par produit (Bar chart horizontal) ---
cout_pertes = pertes.groupby('Designation')['Cout'].sum().sort_values(ascending=True)
cout_pertes.plot(kind='barh', ax=axes[1, 1], color='#E8A75D', edgecolor='black')
axes[1, 1].set_title('Coût Financier des Pertes (Péremptions & Casses)', fontsize=14)
axes[1, 1].set_xlabel('Coût (FCFA)')
axes[1, 1].set_ylabel('')
axes[1, 1].grid(axis='x', linestyle='--', alpha=0.7)


In [ ]:
# --- Subplot 4 : Coût des pertes par produit (Bar chart horizontal) ---
cout_pertes = pertes.groupby('Designation')['Cout'].sum().sort_values(ascending=True)
cout_pertes.plot(kind='barh', ax=axes[1, 1], color='#E8A75D', edgecolor='black')
axes[1, 1].set_title('Coût Financier des Pertes (Péremptions & Casses)', fontsize=14)
axes[1, 1].set_xlabel('Coût (FCFA)')
axes[1, 1].set_ylabel('')
axes[1, 1].grid(axis='x', linestyle='--', alpha=0.7)


In [ ]:
# Ajustement de la mise en page pour éviter que les textes se chevauchent
plt.tight_layout()
plt.subplots_adjust(top=0.90) # Laisser de l'espace pour le titre principal

# Affichage du dashboard
plt.show()